# 03 · Schema Build — Star Schema

We model the cleaned trips as a **star schema**: one fact table + conformed dimensions.
This is the warehouse-portable shape — the same DDL lifts to Snowflake/BigQuery.

```
          dim_date            dim_zone (×2: pickup, dropoff)
              \                 /
               \               /
                +--- fact_trips ---+
               /
        dim_payment
```

We persist to a **file-based DuckDB** so the warehouse survives between notebooks.

In [ ]:
import duckdb
con = duckdb.connect('../warehouse.duckdb')   # persisted file
con.sql("INSTALL httpfs; LOAD httpfs;")

BASE = 'https://d37ci6vzurychx.cloudfront.net/trip-data'
SRC  = f'{BASE}/yellow_tripdata_2023-01.parquet'   # full year: ...-2023-*.parquet
ZONES = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

con.sql(f"""
  CREATE OR REPLACE VIEW trips_clean AS
  SELECT * FROM read_parquet('{SRC}')
  WHERE fare_amount > 0 AND trip_distance > 0
    AND tpep_dropoff_datetime > tpep_pickup_datetime
    AND passenger_count BETWEEN 1 AND 6
    AND EXTRACT(YEAR FROM tpep_pickup_datetime) = 2023
""")
print('Connected to ../warehouse.duckdb')

## 1. `dim_payment` — static encoding
From the brief: 1=Credit, 2=Cash, 3=No charge, 4=Dispute (5=Unknown, 6=Voided also appear in raw data).

In [ ]:
con.sql("""
  CREATE OR REPLACE TABLE dim_payment AS
  SELECT * FROM (VALUES
    (1,'Credit card'), (2,'Cash'), (3,'No charge'),
    (4,'Dispute'), (5,'Unknown'), (6,'Voided trip')
  ) AS t(payment_type, payment_desc)
""")
con.sql("SELECT * FROM dim_payment").df()

## 2. `dim_zone` — geography from the lookup CSV

In [ ]:
con.sql(f"""
  CREATE OR REPLACE TABLE dim_zone AS
  SELECT LocationID AS location_id, Borough AS borough,
         Zone AS zone, service_zone
  FROM read_csv_auto('{ZONES}')
""")
con.sql("SELECT * FROM dim_zone LIMIT 5").df()

## 3. `dim_date` — one row per calendar day of 2023
A conformed date dimension lets analysis slice by month/weekday/weekend without re-deriving it each query.

In [ ]:
con.sql("""
  CREATE OR REPLACE TABLE dim_date AS
  SELECT
    CAST(d AS DATE)                 AS date_key,
    EXTRACT(year   FROM d)          AS year,
    EXTRACT(month  FROM d)          AS month,
    EXTRACT(day    FROM d)          AS day,
    EXTRACT(dow    FROM d)          AS day_of_week,
    dayname(d)                      AS day_name,
    monthname(d)                    AS month_name,
    (EXTRACT(dow FROM d) IN (0,6))  AS is_weekend
  FROM generate_series(DATE '2023-01-01', DATE '2023-12-31', INTERVAL 1 DAY) AS s(d)
""")
con.sql("SELECT * FROM dim_date LIMIT 5").df()

## 4. `fact_trips` — grain = one taxi trip
Foreign keys to the dimensions; measures (distance, fare, tip, total) and a derived duration.

In [ ]:
con.sql("""
  CREATE OR REPLACE TABLE fact_trips AS
  SELECT
    row_number() OVER ()                                   AS trip_id,
    CAST(tpep_pickup_datetime AS DATE)                     AS date_key,     -- -> dim_date
    PULocationID                                           AS pu_location_id, -- -> dim_zone
    DOLocationID                                           AS do_location_id, -- -> dim_zone
    payment_type,                                                            -- -> dim_payment
    tpep_pickup_datetime, tpep_dropoff_datetime,
    date_diff('minute', tpep_pickup_datetime, tpep_dropoff_datetime) AS duration_min,
    passenger_count, trip_distance,
    fare_amount, tip_amount, total_amount
  FROM trips_clean
""")
con.sql("SELECT COUNT(*) AS fact_rows FROM fact_trips").df()

## 5. Verify the joins
Every fact row should resolve against its dimensions (orphan FKs = modeling bug).

In [ ]:
con.sql("""
  SELECT
    (SELECT COUNT(*) FROM fact_trips f LEFT JOIN dim_zone z ON f.pu_location_id=z.location_id WHERE z.location_id IS NULL) AS orphan_pickup_zones,
    (SELECT COUNT(*) FROM fact_trips f LEFT JOIN dim_date d ON f.date_key=d.date_key WHERE d.date_key IS NULL)            AS orphan_dates,
    (SELECT COUNT(*) FROM fact_trips f LEFT JOIN dim_payment p ON f.payment_type=p.payment_type WHERE p.payment_type IS NULL) AS orphan_payments
""").df()

In [ ]:
con.sql("SHOW TABLES").df()

## Takeaways
- Star schema persisted to `../warehouse.duckdb`: `fact_trips` + `dim_zone`, `dim_date`, `dim_payment`.
- Grain is one trip; dimensions are conformed and join-verified.
- The DDL is standard SQL → lifts to Snowflake/BigQuery unchanged.
- Next: **`04_analysis`** — answer business questions off this schema.